In [2]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

# Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

print("Core imports successful!")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")


# ChemProp import (may take a few seconds)
try:
    import chemprop
    print(f"ChemProp version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True
except ImportError:
    print("ChemProp not installed. Install with: pip install chemprop")
    CHEMPROP_AVAILABLE = False

/home/bella/.local/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Core imports successful!
RDKit version: 2025.09.6
ChemProp version: 2.2.2


In [3]:
# RF + Morgan fingerprintings

def compute_morgan_fingerprint(mol: Chem.Mol, radius: int = 2, n_bits: int = 2048) -> np.ndarray:
    """Compute Morgan (ECFP) fingerprint.
    
    Args:
        mol: RDKit Mol object
        radius: Fingerprint radius (2 == ECFP4)
        n_bits: Number of bits (aka the 'length' of the fingerprint)
        
    Returns:
        numpy array of shape (n_bits,)
    """
    fingerpint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    fingerpint = np.array(fingerpint)
    return fingerpint

def train_random_forest(
    X_train,y_train,X_val,y_val,X_test,y_test,
    n_estimators: int = 100,
    random_state: int = 42) -> Dict:


    model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state) #defining the model with the wanted hyperparameters
    model.fit(X_train, y_train) #fitting the model to the hyperparameters and the training data
    
    # Predictions
    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, pred_train))
    val_rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
    test_r2 = r2_score(y_test, pred_test)
    test_rho, _ = spearmanr(y_test, pred_test)
    
    return {
        'model': model,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'test_rho': test_rho,
        'test_predictions': pred_test,
        'test_measured': y_test
    }

In [4]:
#Importing the datasets

lipo_df_test = pd.read_csv('../lipo_splits/lipo_test.csv')
lipo_df_train = pd.read_csv('../lipo_splits/lipo_train.csv')
lipo_df_val = pd.read_csv('../lipo_splits/lipo_val.csv')


In [5]:
# Convert SMILES to RDKit Mol objects
lipo_df_train['mol2'] = lipo_df_train['smiles'].apply(Chem.MolFromSmiles)
lipo_df_val['mol2'] = lipo_df_val['smiles'].apply(Chem.MolFromSmiles)
lipo_df_test['mol2'] = lipo_df_test['smiles'].apply(Chem.MolFromSmiles)

lipo_df_train['mol'] = lipo_df_train['mol2']
lipo_df_val['mol'] = lipo_df_val['mol2']
lipo_df_test['mol'] = lipo_df_test['mol2']

In [6]:


X_train = np.array([compute_morgan_fingerprint(mol) for mol in lipo_df_train['mol']])
y_train = lipo_df_train['exp'].values
X_val = np.array([compute_morgan_fingerprint(mol) for mol in lipo_df_val['mol']])
y_val = lipo_df_val['exp'].values
X_test = np.array([compute_morgan_fingerprint(mol) for mol in lipo_df_test['mol']])
y_test = lipo_df_test['exp'].values

[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerator
[13:34:55] DEPRECATION WARNING: please use MorganGenerat

In [7]:
rf_morgan_scaffold = train_random_forest(X_train,y_train,X_val,y_val,X_test,y_test)
print(f"  Test RMSE: {rf_morgan_scaffold['test_rmse']:.3f}, R²: {rf_morgan_scaffold['test_r2']:.3f}, ρ: {rf_morgan_scaffold['test_rho']:.3f}")

  Test RMSE: 0.870, R²: 0.431, ρ: 0.649


Now same process on the other datasets :

In [ ]:
#Importing the datasets for qm7

qm7_df_test = pd.read_csv('../qm7_splits/qm7_test.csv')
qm7_df_train = pd.read_csv('../qm7_splits/qm7_train.csv')
qm7_df_val = pd.read_csv('../qm7_splits/qm7_val.csv')


# Convert SMILES to RDKit Mol objects
qm7_df_train['mol2'] = qm7_df_train['smiles'].apply(Chem.MolFromSmiles)
qm7_df_val['mol2'] = qm7_df_val['smiles'].apply(Chem.MolFromSmiles)
qm7_df_test['mol2'] = qm7_df_test['smiles'].apply(Chem.MolFromSmiles)

qm7_df_train['mol'] = qm7_df_train['mol2']
qm7_df_val['mol'] = qm7_df_val['mol2']
qm7_df_test['mol'] = qm7_df_test['mol2']



X_train_qm7 = np.array([compute_morgan_fingerprint(mol) for mol in qm7_df_train['mol']])
y_train_qm7 = qm7_df_train['u0_atom'].values
X_val_qm7 = np.array([compute_morgan_fingerprint(mol) for mol in qm7_df_val['mol']])
y_val_qm7 = qm7_df_val['u0_atom'].values
X_test_qm7 = np.array([compute_morgan_fingerprint(mol) for mol in qm7_df_test['mol']])
y_test_qm7 = qm7_df_test['u0_atom'].values

rf_morgan_scaffold_qm7 = train_random_forest(X_train_qm7,y_train_qm7,X_val_qm7,y_val_qm7,X_test_qm7,y_test_qm7)
print(f"  Test RMSE: {rf_morgan_scaffold_qm7['test_rmse']:.3f}, R²: {rf_morgan_scaffold_qm7['test_r2']:.3f}, ρ: {rf_morgan_scaffold_qm7['test_rho']:.3f}")

[13:37:44] WARNING: not removing hydrogen atom without neighbors
[13:37:44] WARNING: not removing hydrogen atom without neighbors
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use MorganGenerator
[13:37:45] DEPRECATION WARNING: please use M

  Test RMSE: 192.306, R²: 0.239, ρ: 0.559


In [17]:
dr_df_test

,Unnamed: 0,neut-smiles,SD,DR,mol,scaffold
0,308853,O=[N+]([O-])c1cc(Br)c2nsnc2c1N1CCOCC1,34.31,4.561157,<rdkit.Chem.rdchem.Mol object at 0x16acb6500>,c1cc(N2CCOCC2)c2nsnc2c1
1,279645,O=C(NCCCN1CCOCC1)C(=O)NN=Cc1ccccc1O,71.52,4.950394,<rdkit.Chem.rdchem.Mol object at 0x16ac79e70>,O=C(NCCCN1CCOCC1)C(=O)NN=Cc1ccccc1
2,96061,CCc1ccccc1NC(=O)CSc1nc2[nH]nc(C)c2c(=N)n1-c1cc...,42.17,4.000000,<rdkit.Chem.rdchem.Mol object at 0x16ac8fdf0>,N=c1c2cn[nH]c2nc(SCC(=O)Nc2ccccc2)n1-c1ccccc1
3,213713,Cc1ccc(S(=O)(=O)N2CC(O)CC2C(=O)OCC(=O)Nc2ccc(N...,34.67,4.202338,<rdkit.Chem.rdchem.Mol object at 0x16acb5cb0>,O=C(COC(=O)C1CCCN1S(=O)(=O)c1ccccc1)Nc1ccc(N2C...
4,78668,CCOC(=O)c1c[nH]c(-n2[nH]c(C)c(Cc3ccccc3)c2=O)n...,89.53,5.468393,<rdkit.Chem.rdchem.Mol object at 0x165e4b610>,O=c1cc[nH]c(-n2[nH]cc(Cc3ccccc3)c2=O)n1
...,...,...,...,...,...,...
77,1819,C=C1c2cc(O)c(O)cc2Oc2cc(O)c(O)cc21,33.36,4.515544,<rdkit.Chem.rdchem.Mol object at 0x16acc1310>,C=C1c2ccccc2Oc2ccccc21
78,7578,CC(=NNC(=S)N1CCCCCC1)c1ccccn1,94.04,5.523024,<rdkit.Chem.rdchem.Mol object at 0x165e4a7a0>,S=C(NN=Cc1ccccn1)N1CCCCCC1
79,157767,COc1ccc(Nc2ccc(S(=O)(=O)N3CCCCCC3)cc2[N+](=O)[...,48.55,4.341226,<rdkit.Chem.rdchem.Mol object at 0x16ac8c3c0>,O=S(=O)(c1ccc(Nc2ccccc2)cc1)N1CCCCCC1
80,124833,COCc1cc(=O)[nH]c(Nc2nc3ccc(C)cc3o2)n1,61.00,4.793444,<rdkit.Chem.rdchem.Mol object at 0x16ac843c0>,O=c1ccnc(Nc2nc3ccccc3o2)[nH]1


In [19]:
#Importing the datasets for dr

dr_df_test = pd.read_csv('../data/data_splits/data/dr_splits/dr_test.csv')
dr_df_train = pd.read_csv('../data/data_splits/data/dr_splits/dr_train.csv')
dr_df_val = pd.read_csv('../data/data_splits/data/dr_splits/dr_val.csv')


# Convert SMILES to RDKit Mol objects
dr_df_train['mol2'] = dr_df_train['neut-smiles'].apply(Chem.MolFromSmiles)
dr_df_val['mol2'] = dr_df_val['neut-smiles'].apply(Chem.MolFromSmiles)
dr_df_test['mol2'] = dr_df_test['neut-smiles'].apply(Chem.MolFromSmiles)

dr_df_train['mol'] = dr_df_train['mol2']
dr_df_val['mol'] = dr_df_val['mol2']
dr_df_test['mol'] = dr_df_test['mol2']



X_train_dr = np.array([compute_morgan_fingerprint(mol) for mol in dr_df_train['mol']])
y_train_dr = dr_df_train['DR'].values
X_val_dr = np.array([compute_morgan_fingerprint(mol) for mol in dr_df_val['mol']])
y_val_dr = dr_df_val['DR'].values
X_test_dr = np.array([compute_morgan_fingerprint(mol) for mol in dr_df_test['mol']])
y_test_dr = dr_df_test['DR'].values

rf_morgan_scaffold_dr = train_random_forest(X_train_dr,y_train_dr,X_val_dr,y_val_dr,X_test_dr,y_test_dr)
print(f"  Test RMSE: {rf_morgan_scaffold_dr['test_rmse']:.3f}, R²: {rf_morgan_scaffold_dr['test_r2']:.3f}, ρ: {rf_morgan_scaffold_dr['test_rho']:.3f}")


[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerator
[14:10:09] DEPRECATION WARNING: please use MorganGenerat

  Test RMSE: 0.401, R²: 0.256, ρ: 0.503
